# E2 · What explainable boosting leans on

The explanations of notebook 12, now with explainable boosting beside the three core models.
Its shares come from its own curves, exactly; a pair term's contribution is split equally
between its two predictors. The **shape functions** below are the curves themselves: what the
model adds to a woman's log-odds at each value of a predictor, which is what makes the model
readable.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

In [ ]:
repro.run_script("run_explainability.py")
# the model cards again, now with all four models, their calibrated versions and the four-model ensembles
repro.run_script("run_model_cards.py", "--library", P.results, "--lineup", "four", *(["--quick"] if QUICK else []))

In [ ]:
agree = pd.read_csv(P.outputs / "explain" / "agreement.csv")
models = ["Logistic regression", "Random Forest", "XGBoost", "Explainable Boosting"]
print(agree[(agree.set == "pool") & (agree.rung == "anc8plus")].head(12)[["predictor", *models, "mean_rank"]].round(3).to_string(index=False))
print(pd.read_csv(P.outputs / "explain" / "concordance.csv").query("set == 'pool' and (a == 'Explainable Boosting' or b == 'Explainable Boosting')").round(2).to_string(index=False))

In [ ]:
import joblib, matplotlib.pyplot as plt
b = joblib.load(P.outputs / "library" / "models" / "pool_anc8plus.joblib")
ebm = b["models"]["Explainable Boosting"].named_steps["model"]
names = list(b["numeric"]) + list(b["categorical"])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, pred in zip(axes, ["v191", "travel_healthpost_log", "age_at_birth"]):
    f = names.index(pred)
    t = next(i for i, feats in enumerate(ebm.term_features_) if tuple(feats) == (f,))
    cuts = np.asarray(ebm.bins_[f][0], dtype=float)
    scores = np.asarray(ebm.term_scores_[t])[1:len(cuts) + 2]
    x = np.r_[cuts[0] - (cuts[1] - cuts[0]), cuts]
    ax.step(x, scores[:len(x)], where="post"); ax.axhline(0, color="grey", lw=0.6)
    ax.set_title(pred); ax.set_ylabel("adds to log-odds" if pred == "v191" else "")
plt.tight_layout(); plt.show()
repro.compare_table(pd.read_csv(P.outputs / "explain" / "importance.csv"), "explain_importance.csv",
                    keys=["set", "rung", "model", "predictor"], value="share", label="every predictor's share, per model")
repro.report()